# HGT experiment: temporal 80/20

Ноутбук содержит только запуск temporal-экспериментов. Весь общий HGT-код импортируется из `hgt_common.py`.

In [8]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd()))

from hgt_common import *

DATA_DIR = Path("..")  # поменяй на Path(".") или абсолютный путь при необходимости
CFG = HGTExperimentConfig(
    k_values=[20],
    target_metrics=["recall", "ndcg"],
    epochs=100,
)
set_seed(CFG.seed)
log_step(f"device: {CFG.device}")
log_step(f"target k_values: {CFG.k_values}")
log_step(f"target metrics: {CFG.target_metrics}")

tables = load_hgt_tables(DATA_DIR)
runner = make_hgt_runner(tables, CFG)


[13:53:18] device: cpu
[13:53:18] target k_values: [20]
[13:53:18] target metrics: ['recall', 'ndcg']
[13:53:18] loaded rates: (948367, 4)
[13:53:18] loaded users: (6040, 4)
[13:53:18] loaded movies: (3433, 7)
[13:53:18] loaded movie_genres: (6408, 2)
[13:53:18] loaded movie_actors: (10285, 2)
[13:53:18] loaded movie_directors: (3676, 2)
[13:53:18] loaded movie_countries: (4765, 2)
[13:53:18] loaded movie_tags: (450912, 2)


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

REPRO_RESULTS_PATH = Path("exp_results/hgt_temporal_reproducibility_runs.csv")
REPRO_SUMMARY_PATH = Path("exp_results/hgt_temporal_reproducibility_summary.csv")


def load_repro_results(path=REPRO_RESULTS_PATH):
    columns = [
        "run_id",
        "seed",
        "stage",
        "threshold",
        "min_pos",
        "k",
        "precision",
        "recall",
        "map",
        "ndcg",
        "mrr",
        "hitrate",
    ]

    if path.exists():
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(columns=columns)
        return df

    return pd.DataFrame(columns=columns)


def save_repro_run(result_df, run_id, seed, stage, threshold, min_pos, path=REPRO_RESULTS_PATH):
    run_df = result_df.copy()

    run_df["run_id"] = run_id
    run_df["seed"] = seed
    run_df["stage"] = stage
    run_df["threshold"] = threshold
    run_df["min_pos"] = min_pos

    old_df = load_repro_results(path)

    combined = pd.concat([old_df, run_df], ignore_index=True)
    combined = combined.drop_duplicates(
        subset=["run_id", "stage", "threshold", "min_pos", "k"],
        keep="last"
    )

    combined = combined.sort_values(["stage", "threshold", "min_pos", "k", "run_id"])
    combined.to_csv(path, index=False)

    return combined


def build_repro_summary(results_df, path=REPRO_SUMMARY_PATH):
    metric_cols = [
        col for col in ["precision", "recall", "map", "ndcg", "mrr", "hitrate"]
        if col in results_df.columns
    ]

    rows = []

    group_cols = ["stage", "threshold", "min_pos", "k"]

    for group_values, group_df in results_df.groupby(group_cols):
        row = dict(zip(group_cols, group_values))
        row["n_runs"] = group_df["run_id"].nunique()

        for metric in metric_cols:
            values = group_df[metric].dropna().astype(float)

            row[f"{metric}_mean"] = values.mean()
            row[f"{metric}_std"] = values.std(ddof=1) if len(values) > 1 else 0.0
            row[f"{metric}_sem"] = values.sem(ddof=1) if len(values) > 1 else 0.0
            row[f"{metric}_ci95"] = (
                1.96 * values.sem(ddof=1) if len(values) > 1 else 0.0
            )

        rows.append(row)

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(path, index=False)

    return summary_df


def run_one_repro_experiment(
    runner,
    run_id,
    stage,
    threshold=5.0,
    min_pos=5,
    base_seed=42,
    results_path=REPRO_RESULTS_PATH,
):
    existing_df = load_repro_results(results_path)

    already_done = (
        not existing_df.empty
        and (
            (existing_df["run_id"] == run_id)
            & (existing_df["stage"] == stage)
            & (existing_df["threshold"] == threshold)
            & (existing_df["min_pos"] == min_pos)
        ).any()
    )

    if already_done:
        print(f"Run {run_id} уже выполнен для stage={stage}. Повторный запуск пропущен.")
        return existing_df, build_repro_summary(existing_df)

    seed = base_seed + run_id

    print("=" * 80)
    print(f"Запуск {run_id}")
    print(f"stage={stage}, threshold={threshold}, min_pos={min_pos}, seed={seed}")
    print("=" * 80)

    set_seed(seed)

    result_df, _ = runner.run_temporal_80_20(
        stage=stage,
        threshold=threshold,
        min_pos=min_pos,
    )

    all_results = save_repro_run(
        result_df=result_df,
        run_id=run_id,
        seed=seed,
        stage=stage,
        threshold=threshold,
        min_pos=min_pos,
        path=results_path,
    )

    summary_df = build_repro_summary(all_results)

    print("Результат запуска сохранён.")
    print(f"Файл результатов: {results_path}")
    print(f"Файл сводки: {REPRO_SUMMARY_PATH}")

    return all_results, summary_df

## Temporal 80/20

Базовый сценарий: один выбранный stage, временное разбиение 80/20, метрики Recall@20 и NDCG@20.

In [12]:
STAGE = "genres_directors_actors_tags"
THRESHOLD = 5.0
MIN_POS = 5

RUN_ID = 5

repro_results, repro_summary = run_one_repro_experiment(
    runner=runner,
    run_id=RUN_ID,
    stage=STAGE,
    threshold=THRESHOLD,
    min_pos=MIN_POS,
    base_seed=42,
)

display(repro_results)
display(repro_summary)

Запуск 5
stage=genres_directors_actors_tags, threshold=5.0, min_pos=5, seed=47
[17:54:10] stage=genres_directors_actors_tags: preparing temporal 80/20 split; threshold=5.0, min_pos=5
[17:54:11] stage=genres_directors_actors_tags: positive stats
users_total    5614.0
min               5.0
p25              13.0
median           24.0
p75              48.0
max             533.0
users_lt5         0.0
users_lt10      868.0
users_lt20     2332.0
dtype: float64
[17:54:11] stage=genres_directors_actors_tags: train/test shapes: train=(168107, 3), test=(44791, 3)
[17:54:11] stage=genres_directors_actors_tags: fit started; k_values=[20]; target_metrics=['recall', 'ndcg']
[17:54:11] stage=genres_directors_actors_tags: building index maps
[17:54:11] context: filtering context tables by train movies
[17:54:11] context sizes after filtering: genres=4908, directors=1747, actors=3675, countries=3952, tags=209972
[17:54:11] stage=genres_directors_actors_tags: building HeteroData
[17:54:11] stage=genres_d

,run_id,seed,stage,threshold,min_pos,k,precision,recall,map,ndcg,mrr,hitrate,n_users_eval
0,4,45,age_group_occupation,5.0,5,20,NaN,0.140901,NaN,0.095777,NaN,NaN,5614.0
1,3,45,genres_actors_age_group_occupation,5.0,5,20,NaN,0.149533,NaN,0.100090,NaN,NaN,5614.0
2,1,43,genres_directors_actors_countries,5.0,5,20,NaN,0.147551,NaN,0.098518,NaN,NaN,5614.0
4,5,47,genres_directors_actors_tags,5.0,5,20,NaN,0.146583,NaN,0.098345,NaN,NaN,5614.0
3,5,47,tags,5.0,5,20,NaN,0.149106,NaN,0.098905,NaN,NaN,5614.0


,stage,threshold,min_pos,k,n_runs,precision_mean,precision_std,precision_sem,precision_ci95,recall_mean,...,ndcg_sem,ndcg_ci95,mrr_mean,mrr_std,mrr_sem,mrr_ci95,hitrate_mean,hitrate_std,hitrate_sem,hitrate_ci95
0,age_group_occupation,5.0,5,20,1,NaN,0.0,0.0,0.0,0.140901,...,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0
1,genres_actors_age_group_occupation,5.0,5,20,1,NaN,0.0,0.0,0.0,0.149533,...,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0
2,genres_directors_actors_countries,5.0,5,20,1,NaN,0.0,0.0,0.0,0.147551,...,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0
3,genres_directors_actors_tags,5.0,5,20,1,NaN,0.0,0.0,0.0,0.146583,...,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0
4,tags,5.0,5,20,1,NaN,0.0,0.0,0.0,0.149106,...,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,0.0
